In [1]:
import os
import subprocess

# 1. Set Kaggle API credentials directly in the environment
os.environ['KAGGLE_USERNAME'] = "hoziyanarachel"
os.environ['KAGGLE_KEY'] = "49a5c172764350186a6ab0856c633e18"

# 2. Ensure kaggle package is installed
try:
    import kaggle
except ImportError:
    print("Installing kaggle library...")
    subprocess.check_call(["pip", "install", "kaggle"])
    import kaggle

# 3. Download and unzip the university chatbot dataset
print("Downloading chatbot dataset...")
kaggle.api.dataset_download_files("niraliivaghani/chatbot-dataset", path=".", unzip=True)

print("Dataset downloaded successfully! 'intents.json' is ready in your project directory.")

Installing kaggle library...
Dataset URL: https://www.kaggle.com/datasets/niraliivaghani/chatbot-dataset
Dataset downloaded successfully! 'intents.json' is ready in your project directory.


In [2]:
import json
import pandas as pd

# Load JSON file
with open("intents.json", "r") as file:
    intents_data = json.load(file)

# Extract intents into a flat list
records = []
for intent in intents_data["intents"]:
    tag = intent["tag"]
    responses = intent["responses"]
    patterns = intent["patterns"]
    for pattern in patterns:
        records.append({
            "tag": tag,
            "pattern": pattern,
            "responses_count": len(responses)
        })

# Create DataFrame
df = pd.DataFrame(records)

# View dataset head
print("--- DATASET HEAD ---")
print(df.head(10))

print("\n--- DATASET SUMMARY ---")
print(f"Total Patterns (Rows): {len(df)}")
print(f"Total Unique Tags/Intents: {df['tag'].nunique()}")

--- DATASET HEAD ---
        tag           pattern  responses_count
0  greeting                Hi                3
1  greeting      How are you?                3
2  greeting  Is anyone there?                3
3  greeting             Hello                3
4  greeting          Good day                3
5  greeting         What's up                3
6  greeting        how are ya                3
7  greeting              heyy                3
8  greeting           whatsup                3
9  greeting        ??? ??? ??                3

--- DATASET SUMMARY ---
Total Patterns (Rows): 405
Total Unique Tags/Intents: 38


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# 1. Clean patterns and drop any empty/invalid rows
df["pattern"] = df["pattern"].astype(str).str.strip()
df = df[df["pattern"] != ""]

# 2. Extract feature input (X) and target labels (y)
X_patterns = df["pattern"].values
y_tags = df["tag"].values

# 3. Vectorize text using TF-IDF (lowercasing and unigram/bigram tokenization)
vectorizer = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(X_patterns)

print("--- DATA PREPARATION COMPLETE ---")
print(f"Total Processed Patterns: {X_tfidf.shape[0]}")
print(f"Vocabulary Size (Unique Terms/N-grams): {X_tfidf.shape[1]}")

--- DATA PREPARATION COMPLETE ---
Total Processed Patterns: 405
Vocabulary Size (Unique Terms/N-grams): 426


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def predict_intent(user_query, threshold=0.20):
    # Vectorize user input
    query_vector = vectorizer.transform([user_query])
    
    # Compute similarity against all stored patterns
    similarities = cosine_similarity(query_vector, X_tfidf).flatten()
    best_idx = np.argmax(similarities)
    best_score = similarities[best_idx]
    
    # Determine output based on confidence threshold
    if best_score >= threshold:
        matched_tag = df.iloc[best_idx]["tag"]
        
        # Retrieve random response for the matched tag
        for intent in intents_data["intents"]:
            if intent["tag"] == matched_tag:
                selected_response = np.random.choice(intent["responses"])
                return matched_tag, best_score, selected_response
    
    return "fallback", best_score, "I am not sure I understand that query regarding university policies. I have flagged this for an advisor."

print("--- MODELING ENGINE CREATED ---")

--- MODELING ENGINE CREATED ---


In [5]:
test_queries = [
    "Hello there",
    "How much do I pay for tuition fees?",
    "Where is the cafeteria located?",
    "How can I contact the head of department?",
    "What time does the university open?"
]

print("--- MODEL EVALUATION ---")
for query in test_queries:
    tag, score, response = predict_intent(query)
    print(f"\nUser Query: '{query}'")
    print(f"Predicted Tag: {tag} | Confidence Score: {score:.4f}")
    print(f"Bot Response: {response}")

--- MODEL EVALUATION ---

User Query: 'Hello there'
Predicted Tag: greeting | Confidence Score: 1.0000
Bot Response: Hi there, how can I help?

User Query: 'How much do I pay for tuition fees?'
Predicted Tag: fees | Confidence Score: 1.0000
Bot Response: For Fee detail visit <a target="_blank" href="LINK"> here</a>

User Query: 'Where is the cafeteria located?'
Predicted Tag: location | Confidence Score: 0.6684
Bot Response: <a target="_blank" href="ADD YOU GOOGLE MAP LINK HERE"> here</a>

User Query: 'How can I contact the head of department?'
Predicted Tag: number | Confidence Score: 1.0000
Bot Response: You can contact at: NUMBER

User Query: 'What time does the university open?'
Predicted Tag: hours | Confidence Score: 0.3896
Bot Response: College is open 8am-5pm Monday-Saturday!


In [6]:
import joblib

joblib.dump(vectorizer, "advising_vectorizer.pkl")
joblib.dump(df, "advising_df.pkl")
print("Model artifacts saved as 'advising_vectorizer.pkl' and 'advising_df.pkl'!")

Model artifacts saved as 'advising_vectorizer.pkl' and 'advising_df.pkl'!
